# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

In [2]:
df = pd.read_csv('AviationData.csv', encoding='latin-1', low_memory=False)
print(df.shape)
df.head()

(88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [4]:
# number of missing values in each column
df.isnull().sum().sort_values(ascending=False)

Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Accident.Number               0
Investigation.Type            0
Event.Id                      0
Event.Date                    0
dtype: i

In [5]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


**What I noticed**
- 88,889 rows and 31 columns. The file needs `encoding='latin-1'` to load.
- `Event.Date` is a string, so it has to be converted to a date.
- `Aircraft.Category` is missing in about 64% of rows. `Schedule` and `Air.carrier` are almost empty.
- The injury columns have 6-14% missing values and a few very large values (one accident has 349 deaths).

## Data Cleaning

### Filtering aircraft and events

**Event.Date:** the data ends in 2022/2023 and a make/model is assumed to retire after 40 years, so I keep events from 1983 onwards.

In [6]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df = df[df['Event.Date'] >= '1983-01-01']
print(df.shape)

(85289, 31)


**Aircraft.Category:** the client only wants airplanes, but most rows have no category. If I dropped them I would lose most of the data, so I fill them in like this:
- Make is converted to upper case so spellings match.
- For each make I count how many of its known records are airplanes and how many are something else (helicopter, glider, ...).
- If a make has more airplane records than other records, then a missing category for that make is set to `Airplane`.
- Rows with a `Turbo Shaft` engine are not filled in because those engines are used in helicopters.
- Rows with no Make are dropped.

In [7]:
df = df.dropna(subset=['Make'])
df['Make'] = df['Make'].str.upper().str.strip()

# 'Unknown' tells us nothing, so treat it as missing
df['Aircraft.Category'] = df['Aircraft.Category'].replace({'Unknown': np.nan, 'UNK': np.nan})

# how many records of each category does each make have?
counts = df.groupby('Make')['Aircraft.Category'].value_counts().unstack(fill_value=0)
other = counts.drop(columns='Airplane').sum(axis=1)
airplane_makes = counts[counts['Airplane'] > other].index

fill_rows = (df['Aircraft.Category'].isnull()) & (df['Make'].isin(airplane_makes)) & (df['Engine.Type'] != 'Turbo Shaft')
print('Missing before:', df['Aircraft.Category'].isnull().sum())
print('Filled in:', fill_rows.sum())
df.loc[fill_rows, 'Aircraft.Category'] = 'Airplane'

df = df[df['Aircraft.Category'] == 'Airplane']
print('Airplane rows:', len(df))

Missing before:

 56530
Filled in: 45995
Airplane rows: 70432


**Amateur.Built:** the client only wants professional builds, so I keep `No`. The few missing values cannot be confirmed as professional builds so they are dropped.

In [8]:
df['Amateur.Built'].value_counts(dropna=False)

Amateur.Built
No     66242
Yes     4101
NaN       89
Name: count, dtype: int64

In [9]:
df = df[df['Amateur.Built'] == 'No']
print(df.shape)

(66242, 31)


### Cleaning and constructing key measurables

**Fatal/serious injury fraction**

Assumptions:
- Number of people on board = fatal + serious + minor + uninjured (this includes crew because the data does not separate them).
- A blank injury count means nobody was in that group, so I fill missing counts with 0. Rows where all four counts are missing are dropped.
- Accidents with 0 people on board are dropped because a fraction cannot be calculated.
- `Fatal.Serious.Fraction` = (fatal + serious) / people on board.

In [10]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

df = df.dropna(subset=injury_cols, how='all')
df[injury_cols] = df[injury_cols].fillna(0)

df['Total.Passengers'] = df[injury_cols].sum(axis=1)
df = df[df['Total.Passengers'] > 0]

df['Fatal.Serious.Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']
df[['Total.Passengers', 'Fatal.Serious.Fraction']].describe()

,Total.Passengers,Fatal.Serious.Fraction
count,65116.000000,65116.000000
mean,7.470084,0.269427
std,31.955867,0.428211
min,1.000000,0.000000
25%,1.000000,0.000000
50%,2.000000,0.000000
75%,3.000000,0.666667
max,699.000000,1.000000


**Aircraft.damage**

The useful values are `Destroyed`, `Substantial` and `Minor`. `Unknown` and missing values do not tell us if the plane was destroyed, so those rows are dropped. The new column `Destroyed` is 1 if the aircraft was destroyed and 0 if not.

In [11]:
print(df['Aircraft.damage'].value_counts(dropna=False))

Aircraft.damage
Substantial    47791
Destroyed      13235
NaN             2011
Minor           2005
Unknown           74
Name: count, dtype: int64


In [12]:
df = df[df['Aircraft.damage'].isin(['Destroyed', 'Substantial', 'Minor'])]
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)
print(df['Destroyed'].value_counts())

Destroyed
0    49796
1    13235
Name: count, dtype: int64


### Investigate the *Make* column

**Cleaning tasks**
1. Remove dots and commas from the names, and remove company endings such as `INC`, `CORP` and `AIRCRAFT CO`.
2. Fix spelling variants and company renames of the same maker (for example `DEHAVILLAND` and `DE HAVILLAND`, `AIRBUS INDUSTRIE` and `AIRBUS`). I found these by looking at the list of makes.
3. Keep only makes with at least 50 accidents.

In [13]:
df['Make'].value_counts().head(40)

Make
CESSNA                            25388
PIPER                             13936
BEECH                              5000
BOEING                             1294
MOONEY                             1256
GRUMMAN                            1055
BELLANCA                            973
AIR TRACTOR                         663
AERONCA                             604
MAULE                               569
CHAMPION                            501
STINSON                             421
AERO COMMANDER                      405
LUSCOMBE                            386
NORTH AMERICAN                      366
TAYLORCRAFT                         364
DE HAVILLAND                        364
ROCKWELL                            334
MCDONNELL DOUGLAS                   311
AYRES                               228
CIRRUS DESIGN CORP                  219
AIR TRACTOR INC                     218
DOUGLAS                             211
GRUMMAN AMERICAN                    211
ERCOUPE (ENG & RESEARCH CORP.)     

In [14]:
df['Make'] = df['Make'].str.replace('.', '', regex=False).str.replace(',', '', regex=False)

# remove company endings like INC, CORP or AIRCRAFT CO (twice, for names like 'AIRCRAFT CO INC')
for i in range(2):
    for ending in [' INC', ' CORP', ' CORPORATION', ' CO', ' COMPANY', ' LTD', ' AIRCRAFT']:
        df['Make'] = df['Make'].str.removesuffix(ending)

make_fixes = {
    'CIRRUS DESIGN CORP': 'CIRRUS', 'CIRRUS DESIGN': 'CIRRUS',
    'DEHAVILLAND': 'DE HAVILLAND',
    'AIRBUS INDUSTRIE': 'AIRBUS',
    'BEECHCRAFT': 'BEECH', 'HAWKER BEECHCRAFT': 'BEECH', 'HAWKER BEECH': 'BEECH',
    'AIR TRACTOR INC': 'AIR TRACTOR',
    'DOUGLAS': 'MCDONNELL DOUGLAS', 'MCDONNELL-DOUGLAS': 'MCDONNELL DOUGLAS',
    'GRUMMAN-SCHWEIZER': 'SCHWEIZER', 'GRUMMAN ACFT ENG COR-SCHWEIZER': 'SCHWEIZER',
    'GULFSTREAM-SCHWEIZER': 'SCHWEIZER', 'GULFSTREAM-SCHWEIZER A/C': 'SCHWEIZER',
    'GRUMMAN AMERICAN AVN': 'GRUMMAN AMERICAN', 'GRUMMAN AMERICAN AVN CORP': 'GRUMMAN AMERICAN',
    'ROCKWELL INTERNATIONAL': 'ROCKWELL',
    'DIAMOND IND': 'DIAMOND', 'DIAMOND INDUSTRIES': 'DIAMOND', 'DIAMOND IND GMBH': 'DIAMOND',
    'AMERICAN CHAMPION': 'CHAMPION', 'AMERICAN CHAMPION (ACAC)': 'CHAMPION',
    'EMBRAER S A': 'EMBRAER', 'EMBRAER SA': 'EMBRAER',
    'GATES LEARJET': 'LEARJET',
    'GULFSTREAM AEROSPACE': 'GULFSTREAM', 'GULFSTREAM AMERICAN': 'GULFSTREAM',
    'ERCOUPE (ENG & RESEARCH CORP)': 'ERCOUPE',
    'RAYTHEON CORPORATE JETS': 'RAYTHEON',
    'MOONEY INTERNATIONAL': 'MOONEY',
    'BRITTEN-NORMAN': 'BRITTEN NORMAN',
    'UNIVERSAL STINSON': 'STINSON',
    'NEW PIPER': 'PIPER',
    'AVIAT AIRCRAFT INC': 'AVIAT',
    'DIAMOND AIRCRAFT IND': 'DIAMOND', 'DIAMOND AIRCRAFT IND INC': 'DIAMOND',
    'AMERICAN CHAMPION AIRCRAFT': 'CHAMPION', 'AMERICAN CHAMPION AIRCRAFT CORP': 'CHAMPION',
    'BOEING STEARMAN': 'BOEING', 'BOEING-STEARMAN': 'BOEING',
    'RAYTHEON AIRCRAFT COMPANY': 'RAYTHEON', 'RAYTHEON AIRCRAFT': 'RAYTHEON',
}
df['Make'] = df['Make'].replace(make_fixes)

print('Unique makes:', df['Make'].nunique())
make_counts = df['Make'].value_counts()
df = df[df['Make'].isin(make_counts[make_counts >= 50].index)]
print('Makes with at least 50 accidents:', df['Make'].nunique())
df['Make'].value_counts()

Unique makes:

 890
Makes with at least 50 accidents: 55


Make
CESSNA                 25434
PIPER                  13989
BEECH                   5076
BOEING                  1347
MOONEY                  1293
GRUMMAN                 1056
BELLANCA                 973
AIR TRACTOR              889
AERONCA                  605
CHAMPION                 601
MAULE                    570
MCDONNELL DOUGLAS        530
DE HAVILLAND             460
CIRRUS                   449
STINSON                  432
AERO COMMANDER           405
ROCKWELL                 403
LUSCOMBE                 386
NORTH AMERICAN           366
TAYLORCRAFT              366
AYRES                    274
GRUMMAN AMERICAN         262
SCHWEIZER                250
ERCOUPE                  232
AVIAT                    230
LEARJET                  197
GULFSTREAM               151
EMBRAER                  150
SWEARINGEN               146
WACO                     138
LAKE                     138
AIRBUS                   138
FAIRCHILD                133
MITSUBISHI               131
DIAMOND  

### Inspect Model column

In [15]:
print('Missing models:', df['Model'].isnull().sum())
df = df.dropna(subset=['Model'])

# make the labels consistent: upper case and no dashes or spaces ('PA-28-180' and 'PA 28 180' become 'PA28180')
df['Model'] = df['Model'].str.upper().str.replace('-', '', regex=False).str.replace(' ', '', regex=False)

# does the same model label belong to more than one make?
makes_per_model = df.groupby('Model')['Make'].nunique()
print((makes_per_model > 1).sum(), 'of', len(makes_per_model), 'model labels are used by more than one make')
makes_per_model.sort_values(ascending=False).head()

Missing models: 13
188 of 3110 model labels are used by more than one make


Model
500      5
G164A    5
AA1A     4
AA1B     4
AA5      4
Name: Make, dtype: int64

Model labels are **not** unique to one make (for example `100` is used by Aero Commander, Rockwell and Beech). So I create `Plane.Type` = Make + Model (like `CESSNA 172N`) to identify a plane type.

In [16]:
df['Plane.Type'] = df['Make'] + ' ' + df['Model']
print('Unique plane types:', df['Plane.Type'].nunique())
df['Plane.Type'].value_counts().head(10)

Unique plane types: 3353


Plane.Type
CESSNA 152       2211
CESSNA 172       1614
CESSNA 172N      1088
PIPER PA28140     925
CESSNA 172M       786
CESSNA 150        742
CESSNA 172P       668
PIPER PA18        663
PIPER PA18150     618
CESSNA 182        608
Name: count, dtype: int64

### Cleaning other columns

In [17]:
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(df[col].value_counts(dropna=False))
    print()

Engine.Type
Reciprocating      51908
NaN                 3066
Turbo Prop          2659
Turbo Fan           1359
Unknown              507
Turbo Jet            377
Turbo Shaft           10
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64

Weather.Condition
VMC    52554
IMC     4808
NaN     1832
UNK      554
Unk      140
Name: count, dtype: int64

Number.of.Engines
1.0    48679
2.0     8153
NaN     2373
4.0      283
3.0      282
0.0      118
Name: count, dtype: int64

Purpose.of.flight
Personal                     35050
Instructional                 8059
Unknown                       3905
Aerial Application            3356
Business                      2960
NaN                           2835
Positioning                   1028
Ferry                          556
Other Work Use                 518
Aerial Observation             381
Executive/corporate            358
Public Aircraft                315
Skydiving                      171
Flight Test                   

**Cleaning tasks**
- `Engine.Type`: `Unknown`, `UNK`, `NONE` and `LR` mean missing, `Geared Turbofan` is the same as `Turbo Fan`.
- `Weather.Condition`: `UNK`, `Unk` and `Unknown` mean missing (the real values are `VMC` and `IMC`).
- `Number.of.Engines`: an airplane cannot have 0 engines, so 0 is a placeholder for missing.
- `Purpose.of.flight`: `Unknown` means missing, and I combine the different spellings of Air Race show, Public Aircraft and Business/Executive.
- `Broad.phase.of.flight`: `Unknown` means missing.

I leave the missing values in these columns.

In [18]:
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'UNK': np.nan, 'NONE': np.nan, 'LR': np.nan,
                                                'Geared Turbofan': 'Turbo Fan'})

df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': np.nan, 'Unk': np.nan, 'Unknown': np.nan})

df['Number.of.Engines'] = df['Number.of.Engines'].replace(0, np.nan)

df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({'Unknown': np.nan,
                                                            'ASHO': 'Air Race show', 'Air Race/show': 'Air Race show',
                                                            'PUBS': 'Public Aircraft', 'PUBL': 'Public Aircraft',
                                                            'Public Aircraft - Federal': 'Public Aircraft',
                                                            'Public Aircraft - Local': 'Public Aircraft',
                                                            'Public Aircraft - State': 'Public Aircraft',
                                                            'Executive/corporate': 'Business'})

df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace({'Unknown': np.nan})

df['Engine.Type'].value_counts(dropna=False)

Engine.Type
Reciprocating    51908
NaN               3574
Turbo Prop        2659
Turbo Fan         1360
Turbo Jet          377
Turbo Shaft         10
Name: count, dtype: int64

### Column Removal

In [19]:
print(len(df))
df.notnull().sum().sort_values().head(10)

59888


Schedule                  7289
Air.carrier              10121
FAR.Description          17526
Longitude                22288
Latitude                 22294
Airport.Code             36588
Airport.Name             38285
Broad.phase.of.flight    43285
Publication.Date         49506
Purpose.of.flight        53148
dtype: int64

I only keep columns with more than 20,000 non-null values. This removes `Schedule` and `Air.carrier`.

In [20]:
non_null = df.notnull().sum()
cols_to_drop = non_null[non_null <= 20000].index
print('Dropping:', list(cols_to_drop))
df = df.drop(columns=cols_to_drop)
df.shape

Dropping: ['FAR.Description', 'Schedule', 'Air.carrier']


(59888, 32)

### Save DataFrame to csv

In [21]:
df.to_csv('cleaned_aviation_data.csv', index=False)
print('Saved', df.shape)

Saved (59888, 32)
